# Lily Vision Phase 2 Multimodal SFT — Modal A100 40GB
**Fine-tunes Lily 1.5B LLM (via QLoRA) + Projector on 64k multimodal CoT samples**

Loads pre-trained Phase 1 aligned projector, applies QLoRA ($r=32, \alpha=64$) to Lily 1.5B LLM across all 7 linear layers, utilizes full 729 visual tokens, and optimizes training via **`LengthGroupedBatchSampler`** (cutting execution time from 3 hours to 20 minutes).

## Cell 1 — Install Dependencies

In [ ]:
# ==============================================================================
# Cell 1 — Dependency Installation (Modal %uv Fast Package Manager)
# ==============================================================================
# Install latest Unsloth framework from GitHub
%uv pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# Install training dependencies & experiment tracking
%uv pip install wandb liger-kernel datasets huggingface_hub -q

# Install flash-attn with --no-build-isolation (to access PyTorch during build)
%uv pip install flash-attn --no-build-isolation -q


## Cell 2 — Configuration & Multimodal SFT Parameters

In [ ]:
# ==============================================================================
# Cell 2 — Multimodal SFT Parameters
# ==============================================================================
import os, torch
HF_USERNAME = "abhinav0231"

MODEL_NAME     = f"{HF_USERNAME}/Lily-1.5b-v0.3"
PROJECTOR_REPO = f"{HF_USERNAME}/lily-vision-projector-v5"
SFT_DATASET    = f"{HF_USERNAME}/lily-vision-sft-64k"
OUTPUT_REPO    = f"{HF_USERNAME}/lily-vision-v0.3"

LEARNING_RATE  = 2e-5
BATCH_SIZE     = 24    # Micro-batch
GRAD_ACCUM     = 2     # Effective Batch = 24 * 2 = 48
LORA_RANK      = 32
LORA_ALPHA     = 64
MAX_SEQ_LENGTH = 3072

print(f"Effective Batch Size: {BATCH_SIZE * GRAD_ACCUM}")

## Cell 3 — `LengthGroupedBatchSampler` Implementation

In [ ]:
# ==============================================================================
# Cell 3 — LengthGroupedBatchSampler Performance Optimization
# Groups multimodal sequences of similar length into mega-batches to minimize
# padding tokens, accelerating training by >8x (3 hours -> 20 minutes).
# ==============================================================================
from torch.utils.data import Sampler
import random

class LengthGroupedBatchSampler(Sampler):
    def __init__(self, lengths, batch_size, mega_batch_mult=50, seed=42):
        self.lengths = lengths
        self.batch_size = batch_size
        self.mega_batch_size = batch_size * mega_batch_mult  # 24 * 50 = 1200 samples
        self.seed = seed

    def __iter__(self):
        indices = list(range(len(self.lengths)))
        random.seed(self.seed)
        random.shuffle(indices)
        mega_batches = [indices[i:i + self.mega_batch_size] for i in range(0, len(indices), self.mega_batch_size)]
        batch_list = []
        for mb in mega_batches:
            mb_sorted = sorted(mb, key=lambda i: self.lengths[i])
            batches = [mb_sorted[i:i + self.batch_size] for i in range(0, len(mb_sorted), self.batch_size)]
            batch_list.extend(batches)
        random.shuffle(batch_list)
        for b in batch_list:
            yield b

print("✅ LengthGroupedBatchSampler defined successfully")